In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, round, datediff, when, to_timestamp

spark = SparkSession.builder.appName("olist_gold").getOrCreate()

df_orders = spark.read.parquet('../data/silver/orders')
df_order_items = spark.read.parquet('../data/silver/order_items')
df_order_payments = spark.read.parquet('../data/silver/order_payments')
df_order_reviews = spark.read.parquet('../data/silver/order_reviews')
df_customers = spark.read.parquet('../data/silver/customers')
df_products = spark.read.parquet('../data/silver/products')
df_sellers = spark.read.parquet('../data/silver/sellers')
df_category = spark.read.parquet('../data/silver/category')
df_geolocation = spark.read.parquet('../data/silver/geolocation')

In [ ]:
df_items_base = df_order_items \
    .join(df_orders, "order_id") \
    .join(df_products, "product_id") \
    .join(df_sellers, "seller_id") \
    .join(df_category, "product_category_name", "left") \
    .join(df_customers, "customer_id")

In [ ]:
df_gold_sales = df_items_base \
    .join(df_customers.select("customer_id", "customer_state").alias("cust"), "customer_id") \
    .select(
        "order_id", "order_item_id", "product_id", "seller_id", "seller_state",
        "price", "freight_value", "order_status",
        "order_purchase_timestamp", "order_delivered_customer_date",
        "order_estimated_delivery_date", "product_category_name_english",
        "customer_id", "cust.customer_state"
    )
df_gold_sales.write.parquet('../data/gold/gold_sales', mode='overwrite')

In [ ]:
df_gold_delivery_satisfaction = df_items_base \
    .join(df_customers.select("customer_id", "customer_state").alias("cust"), "customer_id") \
    .join(df_order_reviews.select("order_id", "review_score", "review_comment_message"), "order_id", "left") \
    .select(
        "order_id", "seller_id", "cust.customer_state",
        "order_purchase_timestamp", "order_delivered_customer_date",
        "order_estimated_delivery_date", "order_status",
        "review_score", "review_comment_message"
    ) \
    .dropDuplicates(["order_id", "seller_id"])

df_gold_delivery_satisfaction.write.parquet('../data/gold/gold_delivery_satisfaction', mode='overwrite')

In [ ]:
df_gold_payments = df_order_payments \
    .join(df_orders.select("order_id", "order_status", "customer_id", "order_purchase_timestamp"), "order_id", "left") \
    .join(df_customers.select("customer_id", "customer_state"), "customer_id", "left") \
    .select(
        "order_id", "payment_type", "payment_value",
        "payment_installments", "payment_sequential",
        "order_status", "order_purchase_timestamp", "customer_state"
    )
df_gold_payments.write.parquet('../data/gold/gold_payments', mode='overwrite')

In [ ]:
df_cust = df_customers.select("customer_id", "customer_state", "customer_city", "customer_zip_code_prefix").alias("cust")
df_geo = df_geolocation.alias("geo")

df_gold_geo = df_items_base \
    .join(df_cust, "customer_id") \
    .join(df_geo, col("cust.customer_zip_code_prefix") == col("geo.geolocation_zip_code_prefix"), "left") \
    .select(
        "order_id", "price", "freight_value",
        "order_purchase_timestamp", "product_category_name_english",
        col("cust.customer_state"), col("cust.customer_city"),
        "seller_state", "seller_city",
        col("geo.geolocation_lat"), col("geo.geolocation_lng")
    )

df_gold_geo.write.parquet('../data/gold/gold_geo', mode='overwrite')

In [ ]:
from pyspark.sql.functions import sum, avg, count, round, desc, trunc, when, datediff

# 1. CA total
df_gold_sales.agg(round(sum("price"), 2).alias("ca_total")).show()



In [ ]:
# 2. CA mensuel
df_gold_sales \
    .withColumn("mois", trunc("order_purchase_timestamp", "MM")) \
    .groupBy("mois") \
    .agg(round(sum("price"), 2).alias("ca_mensuel")) \
    .orderBy("mois") \
    .show(24)

In [ ]:
# 3. Panier moyen
df_gold_sales.groupBy("order_id") \
    .agg(sum("price").alias("total")) \
    .agg(round(avg("total"), 2).alias("panier_moyen")) \
    .show()

In [ ]:
# 4. Top 10 catégories
df_gold_sales.groupBy("product_category_name_english") \
    .agg(round(sum("price"), 2).alias("ca")) \
    .orderBy(desc("ca")) \
    .show(10)

In [ ]:
# 5. Top 10 vendeurs
df_gold_sales.groupBy("seller_id") \
    .agg(round(sum("price"), 2).alias("ca")) \
    .orderBy(desc("ca")) \
    .show(10)

In [ ]:
# 6. Délai moyen de livraison
df_gold_delivery_satisfaction \
    .filter(col("order_delivered_customer_date").isNotNull()) \
    .withColumn("delai", datediff("order_delivered_customer_date", "order_purchase_timestamp")) \
    .agg(round(avg("delai"), 1).alias("delai_moyen_jours")) \
    .show()

In [ ]:
# 7. Taux de retard
df_gold_delivery_satisfaction \
    .filter(col("order_delivered_customer_date").isNotNull()) \
    .withColumn("en_retard", when(
        col("order_delivered_customer_date") > col("order_estimated_delivery_date"), 1
    ).otherwise(0)) \
    .agg(round(avg("en_retard") * 100, 2).alias("taux_retard_pct")) \
    .show()

In [ ]:
# 8. Note moyenne client
df_gold_delivery_satisfaction \
    .agg(round(avg("review_score"), 2).alias("note_moyenne")) \
    .show()

In [ ]:
# 9. Impact des retards sur la satisfaction
df_gold_delivery_satisfaction \
    .filter(col("order_delivered_customer_date").isNotNull()) \
    .withColumn("en_retard", when(
        col("order_delivered_customer_date") > col("order_estimated_delivery_date"), "retard"
    ).otherwise("a_temps")) \
    .groupBy("en_retard") \
    .agg(round(avg("review_score"), 2).alias("note_moyenne")) \
    .show()

In [ ]:
# 10. Répartition des paiements
df_gold_payments \
    .groupBy("payment_type") \
    .agg(
        count("order_id").alias("nb_commandes"),
        round(sum("payment_value"), 2).alias("montant_total")
    ) \
    .orderBy(desc("nb_commandes")) \
    .show()

In [ ]:
# 11. CA par état brésilien
df_gold_sales \
    .groupBy("customer_state") \
    .agg(round(sum("price"), 2).alias("ca")) \
    .orderBy(desc("ca")) \
    .show(10)

In [ ]:
# 12. Taux de commandes par statut
df_gold_sales \
    .groupBy("order_status") \
    .agg(count("order_id").alias("nb_commandes")) \
    .orderBy(desc("nb_commandes")) \
    .show()

In [ ]:
# 13. Note moyenne par catégorie
df_gold_delivery_satisfaction \
    .join(df_gold_sales.select("order_id", "product_category_name_english"), "order_id", "left") \
    .groupBy("product_category_name_english") \
    .agg(round(avg("review_score"), 2).alias("note_moyenne")) \
    .orderBy(desc("note_moyenne")) \
    .show(10)

In [ ]:
# 14. Délai moyen par état
df_gold_delivery_satisfaction \
    .filter(col("order_delivered_customer_date").isNotNull()) \
    .withColumn("delai", datediff("order_delivered_customer_date", "order_purchase_timestamp")) \
    .groupBy("customer_state") \
    .agg(round(avg("delai"), 1).alias("delai_moyen_jours")) \
    .orderBy(desc("delai_moyen_jours")) \
    .show(10)

In [ ]:
# 15. CA par vendeur et état
df_gold_sales \
    .groupBy("seller_id", "seller_state") \
    .agg(round(sum("price"), 2).alias("ca")) \
    .orderBy(desc("ca")) \
    .show(10)